In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
prepared_file_path = r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\prepared_data\EduPro_Prepared.xlsx"

df = pd.read_excel(
    prepared_file_path,
    sheet_name="Prepared_Data"
)

In [3]:
df.shape

(10000, 29)

## Creating a Seprate Featuer Datasets

In [4]:
features = df.copy()    

In [5]:
features.shape
features.head()

,TransactionID,UserID,CourseID,TransactionDate,Amount,PaymentMethod,TeacherID,UserName,UserAge,UserGender,...,TeacherAge,TeacherGender,Expertise,YearsOfExperience,TeacherRating,TransactionYear,TransactionMonth,TransactionDay,TransactionDayOfWeek,TransactionQuarter
0,TT00001,U00003,CR00016,2025-10-25,0.0,PayPal,TC00040,morrisonamanda,33,Female,...,49,Male,Cybersecurity,24,4.58,2025,10,25,5,4
1,TT00002,U00003,CR00037,2025-01-13,0.0,PayPal,TC00040,morrisonamanda,33,Female,...,49,Male,Cybersecurity,24,4.58,2025,1,13,0,1
2,TT00003,U00003,CR00019,2025-03-28,0.0,Bank Transfer,TC00040,morrisonamanda,33,Female,...,49,Male,Cybersecurity,24,4.58,2025,3,28,4,1
3,TT00004,U00004,CR00048,2025-06-02,0.0,Bank Transfer,TC00040,fthornton,23,Female,...,49,Male,Cybersecurity,24,4.58,2025,6,2,0,2
4,TT00005,U00004,CR00060,2025-08-10,0.0,PayPal,TC00042,fthornton,23,Female,...,49,Female,Machine Learning,21,4.97,2025,8,10,6,3


## Feature Engineering Strategy

### PART A — Date Features

In [6]:
features["MonthSin"] = np.sin(
    2 * np.pi * features["TransactionMonth"] / 12
)

features["MonthCos"] = np.cos(
    2 * np.pi * features["TransactionMonth"] / 12
)

In [7]:
features["IsWeekend"] = (
    features["TransactionDayOfWeek"] >= 5
).astype(int)

In [8]:
features["TransactionDayOfWeek"].unique()

array([5, 0, 4, 6, 3, 2, 1])

### PART B — Price Features

In [9]:
features["PriceDifference"] = (
    features["CoursePrice"] - features["Amount"]
)

In [10]:
features["DiscountAmount"] = (
    features["CoursePrice"] - features["Amount"]
)

features["DiscountPercentage"] = np.where(
    features["CoursePrice"] > 0,
    (
        features["DiscountAmount"]
        / features["CoursePrice"]
    ) * 100,
    0
)

In [11]:
features["HasDiscount"] = (
    features["DiscountAmount"] > 0
).astype(int)

In [12]:
features[
    [
        "Amount",
        "CoursePrice",
        "PriceDifference",
        "DiscountPercentage",
        "HasDiscount"
    ]
].head()

,Amount,CoursePrice,PriceDifference,DiscountPercentage,HasDiscount
0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0


### PART C — Course Features

In [13]:
features["CourseDuration"].describe()

count    10000.000000
mean        27.509536
std         16.011616
min          1.200000
25%         12.130000
50%         28.330000
75%         42.700000
max         49.730000
Name: CourseDuration, dtype: float64

In [14]:
features["CourseDurationLog"] = np.log1p(
    features["CourseDuration"]
)

#### Course rating transformation

In [15]:
features["HighRatedCourse"] = (
    features["CourseRating"] >= 4
).astype(int)

In [16]:
features["CourseRating"].describe()

count    10000.000000
mean         3.123277
std          1.156254
min          1.130000
25%          2.140000
50%          3.110000
75%          4.110000
max          4.940000
Name: CourseRating, dtype: float64

#### Course price log

In [17]:
features["CoursePriceLog"] = np.log1p(
    features["CoursePrice"]
)

### PART D — Teacher Features

#### Experienced category

In [18]:
features["ExperienceLevel"] = pd.cut(
    features["YearsOfExperience"],
    bins=[-1, 5, 10, 20, np.inf],
    labels=[
        "Beginner",
        "Intermediate",
        "Experienced",
        "Expert"
    ]
)

#### Highly experienced teacher

In [19]:
features["HighlyExperiencedTeacher"] = (
    features["YearsOfExperience"] >= 10
).astype(int)

#### Highly rated teacher 

In [20]:
features["HighRatedTeacher"] = (
    features["TeacherRating"] >= 4
).astype(int)

### PART E — User Features

#### Age groups

In [21]:
features["UserAgeGroup"] = pd.cut(
    features["UserAge"],
    bins=[0, 18, 25, 35, 45, 60, np.inf],
    labels=[
        "Under18",
        "18-25",
        "26-35",
        "36-45",
        "46-60",
        "60Plus"
    ]
)

#### Adult user flag

In [22]:
features["AdultUser"] = (
    features["UserAge"] >= 18
).astype(int)

### PART F — Interaction Features

#### Price x Rating 

In [23]:
features["PriceRatingInteraction"] = (
    features["CoursePrice"]
    * features["CourseRating"]
)

#### Teacher experience x rating

In [24]:
features["ExperienceRatingInteraction"] = (
    features["YearsOfExperience"]
    * features["TeacherRating"]
)

#### Course duration x price 

In [25]:
features["PricePerDuration"] = np.where(
    features["CourseDuration"] > 0,
    features["CoursePrice"]
    / features["CourseDuration"],
    0
)

#### Teacher-course rating 

In [26]:
features["TeacherCourseRatingInteraction"] = (
    features["TeacherRating"]
    * features["CourseRating"]
)

### PART G — Historical Aggregated Features

In [27]:
features["TransactionDate"] = pd.to_datetime(
    features["TransactionDate"]
)

features = features.sort_values(
    "TransactionDate"
).reset_index(drop=True)

In [28]:
features["CoursePreviousDemand"] = (
    features.groupby("CourseID")
    .cumcount()
)

In [29]:
features["TeacherPreviousTransactions"] = (
    features.groupby("TeacherID")
    .cumcount()
)

### PART H — Categorical Features

In [30]:
categorical_columns = [
    "PaymentMethod",
    "UserGender",
    "CourseCategory",
    "CourseType",
    "CourseLevel",
    "TeacherGender",
    "Expertise"
]

for col in categorical_columns:
    print("\n", col)
    print(features[col].value_counts())


 PaymentMethod
PaymentMethod
PayPal           3389
Credit Card      3333
Bank Transfer    3278
Name: count, dtype: int64

 UserGender
UserGender
Female    5078
Male      4922
Name: count, dtype: int64

 CourseCategory
CourseCategory
Data Science               916
Finance                    864
Web Development            844
Business                   833
Artificial Intelligence    829
Project Management         829
Design                     827
Machine Learning           819
Cybersecurity              819
Digital Marketing          808
Marketing                  806
Programming                806
Name: count, dtype: int64

 CourseType
CourseType
Free    6403
Paid    3597
Name: count, dtype: int64

 CourseLevel
CourseLevel
Beginner        3573
Advanced        3475
Intermediate    2952
Name: count, dtype: int64

 TeacherGender
TeacherGender
Male      5074
Female    4926
Name: count, dtype: int64

 Expertise
Expertise
Cybersecurity              3450
Machine Learning           3303
Digit

### PART I — Remove Leakage-Prone / Non-Predictive Columns

In [31]:
import pandas as pd
import os

# Day 6 prepared dataset
input_path = r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\prepared_data\EduPro_Prepared.xlsx"

# Load Prepared_Data sheet
df = pd.read_excel(
    input_path,
    sheet_name="Prepared_Data"
)

print("Dataset loaded successfully.")
print(f"Shape: {df.shape}")

Dataset loaded successfully.
Shape: (10000, 29)


In [32]:
print("Available columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

Available columns:
1. TransactionID
2. UserID
3. CourseID
4. TransactionDate
5. Amount
6. PaymentMethod
7. TeacherID
8. UserName
9. UserAge
10. UserGender
11. Email
12. CourseName
13. CourseCategory
14. CourseType
15. CourseLevel
16. CoursePrice
17. CourseDuration
18. CourseRating
19. TeacherName
20. TeacherAge
21. TeacherGender
22. Expertise
23. YearsOfExperience
24. TeacherRating
25. TransactionYear
26. TransactionMonth
27. TransactionDay
28. TransactionDayOfWeek
29. TransactionQuarter


In [33]:
# Leakage-prone / non-predictive columns
exclude_columns = [
    "TransactionID",
    "UserID",
    "CourseID",
    "TeacherID",
    "UserName",
    "TeacherName",
    "CourseName",
    "Email"
]

# Check which columns actually exist
existing_exclude_columns = [
    col for col in exclude_columns
    if col in df.columns
]

missing_exclude_columns = [
    col for col in exclude_columns
    if col not in df.columns
]

print("Columns to exclude:")
print(existing_exclude_columns)

if missing_exclude_columns:
    print("\nNot found in dataset:")
    print(missing_exclude_columns)

Columns to exclude:
['TransactionID', 'UserID', 'CourseID', 'TeacherID', 'UserName', 'TeacherName', 'CourseName', 'Email']


In [34]:
# Keep identifiers / tracking information separately
tracking_columns = [
    "TransactionID",
    "UserID",
    "CourseID",
    "TeacherID",
    "UserName",
    "TeacherName",
    "CourseName",
    "Email"
]

existing_tracking_columns = [
    col for col in tracking_columns
    if col in df.columns
]

tracking_df = df[existing_tracking_columns].copy()

print("Tracking dataframe created.")
print(f"Shape: {tracking_df.shape}")

tracking_df.head()

Tracking dataframe created.
Shape: (10000, 8)


,TransactionID,UserID,CourseID,TeacherID,UserName,TeacherName,CourseName,Email
0,TT00001,U00003,CR00016,TC00040,morrisonamanda,Kimberly Miller,Digital Marketing,ganderson@yahoo.com
1,TT00002,U00003,CR00037,TC00040,morrisonamanda,Kimberly Miller,Scrum Essentials,ganderson@yahoo.com
2,TT00003,U00003,CR00019,TC00040,morrisonamanda,Kimberly Miller,Content Marketing,ganderson@yahoo.com
3,TT00004,U00004,CR00048,TC00040,fthornton,Kimberly Miller,AI Ethics,christensencatherine@outlook.com
4,TT00005,U00004,CR00060,TC00042,fthornton,Yolanda Levine,Content Creation,christensencatherine@outlook.com


In [35]:
# Remove non-predictive / identifier columns from modeling data
model_df = df.drop(
    columns=existing_exclude_columns
).copy()

print("Model dataset created.")
print(f"Shape: {model_df.shape}")

Model dataset created.
Shape: (10000, 21)


In [36]:
print("Columns available for modeling:")
for i, col in enumerate(model_df.columns, start=1):
    print(f"{i}. {col}")

Columns available for modeling:
1. TransactionDate
2. Amount
3. PaymentMethod
4. UserAge
5. UserGender
6. CourseCategory
7. CourseType
8. CourseLevel
9. CoursePrice
10. CourseDuration
11. CourseRating
12. TeacherAge
13. TeacherGender
14. Expertise
15. YearsOfExperience
16. TeacherRating
17. TransactionYear
18. TransactionMonth
19. TransactionDay
20. TransactionDayOfWeek
21. TransactionQuarter


In [37]:
print(model_df.columns.tolist())

['TransactionDate', 'Amount', 'PaymentMethod', 'UserAge', 'UserGender', 'CourseCategory', 'CourseType', 'CourseLevel', 'CoursePrice', 'CourseDuration', 'CourseRating', 'TeacherAge', 'TeacherGender', 'Expertise', 'YearsOfExperience', 'TeacherRating', 'TransactionYear', 'TransactionMonth', 'TransactionDay', 'TransactionDayOfWeek', 'TransactionQuarter']


In [38]:
target_columns = [
    "EnrollmentCount",
    "CourseRevenue"
]

existing_targets = [
    col for col in target_columns
    if col in model_df.columns
]

print("Target columns found:")
print(existing_targets)

Target columns found:
[]


In [39]:
# Target dataframe
y = model_df[existing_targets].copy()

# Feature dataframe
X = model_df.drop(
    columns=existing_targets
).copy()

print("Features (X):", X.shape)
print("Targets (y):", y.shape)

Features (X): (10000, 21)
Targets (y): (10000, 0)


In [40]:
print("===== DAY 7 PART I VALIDATION =====")

print("\nOriginal dataset:")
print(df.shape)

print("\nTracking dataframe:")
print(tracking_df.shape)

print("\nFeature dataframe X:")
print(X.shape)

print("\nTarget dataframe y:")
print(y.shape)

print("\nExcluded columns:")
print(existing_exclude_columns)

print("\nTargets:")
print(existing_targets)

===== DAY 7 PART I VALIDATION =====

Original dataset:
(10000, 29)

Tracking dataframe:
(10000, 8)

Feature dataframe X:
(10000, 21)

Target dataframe y:
(10000, 0)

Excluded columns:
['TransactionID', 'UserID', 'CourseID', 'TeacherID', 'UserName', 'TeacherName', 'CourseName', 'Email']

Targets:
[]


In [41]:
# Verify that no excluded columns remain in X
remaining_excluded = [
    col for col in existing_exclude_columns
    if col in X.columns
]

if len(remaining_excluded) == 0:
    print("✅ All identifier/non-predictive columns successfully removed from X.")
else:
    print("❌ These columns are still present:")
    print(remaining_excluded)

✅ All identifier/non-predictive columns successfully removed from X.


In [42]:
output_dir = r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data"

os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "EduPro_Day7_Feature_Selection.xlsx"
)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    X.to_excel(writer, sheet_name="Model_Features", index=False)
    y.to_excel(writer, sheet_name="Targets", index=False)
    tracking_df.to_excel(writer, sheet_name="Tracking_IDs", index=False)

print(f"✅ Day 7 Part I file saved successfully:")
print(output_path)

✅ Day 7 Part I file saved successfully:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Day7_Feature_Selection.xlsx


### PART J — Feature Validation

In [43]:
features.shape

(10000, 50)

In [44]:
features.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 50 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   TransactionID                   10000 non-null  str           
 1   UserID                          10000 non-null  str           
 2   CourseID                        10000 non-null  str           
 3   TransactionDate                 10000 non-null  datetime64[us]
 4   Amount                          10000 non-null  float64       
 5   PaymentMethod                   10000 non-null  str           
 6   TeacherID                       10000 non-null  str           
 7   UserName                        10000 non-null  str           
 8   UserAge                         10000 non-null  int64         
 9   UserGender                      10000 non-null  str           
 10  Email                           10000 non-null  str           
 11  CourseName    

In [45]:
features.isnull().sum().sum()

np.int64(0)

In [46]:
features.duplicated().sum()

np.int64(0)

In [47]:
features.describe()

,TransactionDate,Amount,UserAge,CoursePrice,CourseDuration,CourseRating,TeacherAge,YearsOfExperience,TeacherRating,TransactionYear,...,CoursePriceLog,HighlyExperiencedTeacher,HighRatedTeacher,AdultUser,PriceRatingInteraction,ExperienceRatingInteraction,PricePerDuration,TeacherCourseRatingInteraction,CoursePreviousDemand,TeacherPreviousTransactions
count,10000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.0,...,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.0000,10000.000000
mean,2025-07-01 03:50:32.640000,91.132347,24.965700,91.132347,27.509536,3.123277,44.885000,16.024100,4.087769,2025.0,...,1.846283,0.674800,0.682000,0.853100,279.099890,73.203755,7.401297,12.765285,83.2960,942.368400
min,2025-01-01 00:00:00,0.000000,15.000000,0.000000,1.200000,1.130000,27.000000,1.000000,1.050000,2025.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.050000,0.000000,1.186500,0.0000,0.000000
25%,2025-04-02 00:00:00,0.000000,20.000000,0.000000,12.130000,2.140000,44.000000,7.000000,3.460000,2025.0,...,0.000000,0.000000,0.000000,1.000000,0.000000,25.970000,0.000000,7.891600,41.0000,44.000000
50%,2025-06-30 00:00:00,0.000000,25.000000,0.000000,28.330000,3.110000,49.000000,21.000000,4.580000,2025.0,...,0.000000,1.000000,1.000000,1.000000,0.000000,104.370000,0.000000,12.317200,83.0000,542.500000
75%,2025-09-28 00:00:00,119.040000,30.000000,119.040000,42.700000,4.110000,49.000000,24.000000,4.970000,2025.0,...,4.787825,1.000000,1.000000,1.000000,302.437600,109.920000,5.085006,17.892000,124.2500,1792.250000
max,2025-12-30 00:00:00,490.900000,35.000000,490.900000,49.730000,4.940000,50.000000,24.000000,4.970000,2025.0,...,6.198275,1.000000,1.000000,1.000000,2238.607200,109.920000,101.377193,24.054800,195.0000,3060.000000
std,NaN,152.063524,6.051858,152.063524,16.011616,1.156254,6.988566,8.521783,1.069542,0.0,...,2.577054,0.468473,0.465723,0.354024,545.955887,43.599751,18.155322,5.899103,48.9127,995.485483


#### Checking infinte values

In [48]:
np.isinf(
    features.select_dtypes(include=np.number)
).sum().sum()

np.int64(0)

In [49]:
features.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

,TransactionID,UserID,CourseID,TransactionDate,Amount,PaymentMethod,TeacherID,UserName,UserAge,UserGender,...,HighlyExperiencedTeacher,HighRatedTeacher,UserAgeGroup,AdultUser,PriceRatingInteraction,ExperienceRatingInteraction,PricePerDuration,TeacherCourseRatingInteraction,CoursePreviousDemand,TeacherPreviousTransactions
0,TT06004,U02064,CR00050,2025-01-01,490.90,Bank Transfer,TC00040,debra55,35,Male,...,1,1,26-35,1,2233.5950,109.92,65.019868,20.8390,0,0
1,TT03662,U02547,CR00021,2025-01-01,0.00,Bank Transfer,TC00016,ryanallen,30,Female,...,0,0,26-35,1,0.0000,2.92,0.000000,10.5120,0,0
2,TT03648,U02508,CR00009,2025-01-01,0.00,PayPal,TC00051,athomas,32,Male,...,0,0,26-35,1,0.0000,3.54,0.000000,7.9827,0,0
3,TT05429,U01144,CR00021,2025-01-01,0.00,PayPal,TC00016,fsanders,29,Female,...,0,0,26-35,1,0.0000,2.92,0.000000,10.5120,1,1
4,TT03309,U01657,CR00022,2025-01-01,0.00,Bank Transfer,TC00010,lopezjesse,22,Female,...,0,0,18-25,1,0.0000,2.18,0.000000,7.9570,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,TT02816,U00336,CR00048,2025-12-30,0.00,Bank Transfer,TC00027,milescaitlin,19,Male,...,0,0,18-25,1,0.0000,4.65,0.000000,5.6420,185,132
9996,TT00979,U01553,CR00027,2025-12-30,0.00,Bank Transfer,TC00042,melinda52,27,Male,...,1,1,26-35,1,0.0000,104.37,0.000000,8.9957,151,3059
9997,TT02040,U00887,CR00060,2025-12-30,0.00,Credit Card,TC00019,joseph48,17,Male,...,0,0,Under18,0,0.0000,27.20,0.000000,7.2760,164,34
9998,TT08088,U02683,CR00030,2025-12-30,0.00,Bank Transfer,TC00042,jake76,28,Male,...,1,1,26-35,1,0.0000,104.37,0.000000,8.5981,168,3060


## Feature Inventory 

In [50]:
feature_summary = pd.DataFrame({
    "Feature": features.columns,
    "DataType": features.dtypes.astype(str).values,
    "MissingValues": features.isnull().sum().values,
    "UniqueValues": features.nunique().values
})

feature_summary

,Feature,DataType,MissingValues,UniqueValues
0,TransactionID,str,0,10000
1,UserID,str,0,3000
2,CourseID,str,0,60
3,TransactionDate,datetime64[us],0,358
4,Amount,float64,0,23
5,PaymentMethod,str,0,3
6,TeacherID,str,0,60
7,UserName,str,0,3000
8,UserAge,int64,0,21
9,UserGender,str,0,2


## Correlation Analysis

In [51]:
numeric_features = features.select_dtypes(
    include=np.number
)

correlation_matrix = numeric_features.corr()

correlation_matrix

,Amount,UserAge,CoursePrice,CourseDuration,CourseRating,TeacherAge,YearsOfExperience,TeacherRating,TransactionYear,TransactionMonth,...,CoursePriceLog,HighlyExperiencedTeacher,HighRatedTeacher,AdultUser,PriceRatingInteraction,ExperienceRatingInteraction,PricePerDuration,TeacherCourseRatingInteraction,CoursePreviousDemand,TeacherPreviousTransactions
Amount,1.000000,-0.004630,1.000000,-0.088745,-0.031465,0.056525,0.024223,-0.012513,NaN,-0.007204,...,0.909361,0.043589,0.008202,-0.012024,0.911644,0.019731,0.652257,-0.030854,-0.020388,0.004856
UserAge,-0.004630,1.000000,-0.004630,-0.010838,-0.003945,-0.001900,0.002436,-0.004712,NaN,0.020713,...,-0.005950,0.000122,-0.002203,0.618386,-0.009504,0.001086,0.004980,-0.009765,0.017323,0.013532
CoursePrice,1.000000,-0.004630,1.000000,-0.088745,-0.031465,0.056525,0.024223,-0.012513,NaN,-0.007204,...,0.909361,0.043589,0.008202,-0.012024,0.911644,0.019731,0.652257,-0.030854,-0.020388,0.004856
CourseDuration,-0.088745,-0.010838,-0.088745,1.000000,0.198433,-0.008243,-0.015613,-0.003035,NaN,0.000670,...,-0.019538,-0.020738,-0.030072,-0.009004,-0.088792,-0.014505,-0.386951,0.158181,-0.013610,-0.006695
CourseRating,-0.031465,-0.003945,-0.031465,0.198433,1.000000,-0.010420,-0.007015,-0.001577,NaN,-0.009554,...,-0.108404,-0.010187,0.006198,-0.009906,0.196036,-0.007914,-0.107405,0.803026,0.038993,-0.020180
TeacherAge,0.056525,-0.001900,0.056525,-0.008243,-0.010420,1.000000,0.740179,0.707718,NaN,-0.013189,...,0.060374,0.714865,0.638435,-0.011073,0.035160,0.755146,0.087772,0.387143,-0.011991,0.533399
YearsOfExperience,0.024223,0.002436,0.024223,-0.015613,-0.007015,0.740179,1.000000,0.845004,NaN,-0.000225,...,0.028576,0.932665,0.863641,-0.012981,0.014815,0.992337,0.012722,0.471184,-0.001013,0.685483
TeacherRating,-0.012513,-0.004712,-0.012513,-0.003035,-0.001577,0.707718,0.845004,1.000000,NaN,-0.008438,...,0.000801,0.781741,0.864945,-0.012297,-0.008974,0.876048,-0.010031,0.556873,-0.009795,0.581951
TransactionYear,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TransactionMonth,-0.007204,0.020713,-0.007204,0.000670,-0.009554,-0.013189,-0.000225,-0.008438,NaN,1.000000,...,-0.006356,0.003912,-0.003074,0.012250,-0.012879,-0.003350,-0.002148,-0.014569,0.981706,0.543094


In [52]:
selected_features = [
    "Amount",
    "CoursePrice",
    "CourseRating",
    "TeacherRating",
    "YearsOfExperience"
]

correlation_matrix.loc[selected_features, selected_features]

,Amount,CoursePrice,CourseRating,TeacherRating,YearsOfExperience
Amount,1.000000,1.000000,-0.031465,-0.012513,0.024223
CoursePrice,1.000000,1.000000,-0.031465,-0.012513,0.024223
CourseRating,-0.031465,-0.031465,1.000000,-0.001577,-0.007015
TeacherRating,-0.012513,-0.012513,-0.001577,1.000000,0.845004
YearsOfExperience,0.024223,0.024223,-0.007015,0.845004,1.000000


## Checking Highly Correlated Features

In [53]:
corr_pairs = (
    correlation_matrix
    .abs()
    .unstack()
    .sort_values(ascending=False)
)

corr_pairs.head(20)

TeacherPreviousTransactions     TeacherPreviousTransactions       1.0
Amount                          Amount                            1.0
CoursePreviousDemand            CoursePreviousDemand              1.0
UserAge                         UserAge                           1.0
TeacherCourseRatingInteraction  TeacherCourseRatingInteraction    1.0
PricePerDuration                PricePerDuration                  1.0
ExperienceRatingInteraction     ExperienceRatingInteraction       1.0
PriceRatingInteraction          PriceRatingInteraction            1.0
AdultUser                       AdultUser                         1.0
HighRatedTeacher                HighRatedTeacher                  1.0
HighlyExperiencedTeacher        HighlyExperiencedTeacher          1.0
CoursePriceLog                  CoursePriceLog                    1.0
HighRatedCourse                 HighRatedCourse                   1.0
CourseDurationLog               CourseDurationLog                 1.0
IsWeekend           

## Saving the Featuring Datasets (Day7)

In [54]:
feature_folder = Path(
    r"D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data"
)

feature_folder.mkdir(
    parents=True,
    exist_ok=True
)

In [55]:
feature_file_path = (
    feature_folder
    / "EduPro_Feature_Engineered.xlsx"
)

In [56]:
with pd.ExcelWriter(
    feature_file_path,
    engine="openpyxl"
) as writer:

    features.to_excel(
        writer,
        sheet_name="Feature_Engineered",
        index=False
    )

    feature_summary.to_excel(
        writer,
        sheet_name="Feature_Summary",
        index=False
    )

In [57]:
print(
    f"Feature Engineering output saved at:\n"
    f"{feature_file_path}"
)

Feature Engineering output saved at:
D:\Data Analytics Project\EduPro_Predictive_Modeling\data\feature_data\EduPro_Feature_Engineered.xlsx


## Final Validation after saving 

In [58]:
check_features = pd.read_excel(
    feature_file_path,
    sheet_name="Feature_Engineered"
)

In [59]:
print("Rows:", check_features.shape[0])
print("Columns:", check_features.shape[1])
print("Missing values:", check_features.isnull().sum().sum())
print("Duplicate rows:", check_features.duplicated().sum())

Rows: 10000
Columns: 50
Missing values: 0
Duplicate rows: 0


In [60]:
check_features.head()

,TransactionID,UserID,CourseID,TransactionDate,Amount,PaymentMethod,TeacherID,UserName,UserAge,UserGender,...,HighlyExperiencedTeacher,HighRatedTeacher,UserAgeGroup,AdultUser,PriceRatingInteraction,ExperienceRatingInteraction,PricePerDuration,TeacherCourseRatingInteraction,CoursePreviousDemand,TeacherPreviousTransactions
0,TT06004,U02064,CR00050,2025-01-01,490.9,Bank Transfer,TC00040,debra55,35,Male,...,1,1,26-35,1,2233.595,109.92,65.019868,20.8390,0,0
1,TT03662,U02547,CR00021,2025-01-01,0.0,Bank Transfer,TC00016,ryanallen,30,Female,...,0,0,26-35,1,0.000,2.92,0.000000,10.5120,0,0
2,TT03648,U02508,CR00009,2025-01-01,0.0,PayPal,TC00051,athomas,32,Male,...,0,0,26-35,1,0.000,3.54,0.000000,7.9827,0,0
3,TT05429,U01144,CR00021,2025-01-01,0.0,PayPal,TC00016,fsanders,29,Female,...,0,0,26-35,1,0.000,2.92,0.000000,10.5120,1,1
4,TT03309,U01657,CR00022,2025-01-01,0.0,Bank Transfer,TC00010,lopezjesse,22,Female,...,0,0,18-25,1,0.000,2.18,0.000000,7.9570,0,0
